**Séance :** 01  
**Nom :** FANOU Précieux  
**Porteur # :** P2  
**Étape assignée :** 1.2 --> 1.3 --> 1.5 --> 1.10

# Étape 1.5 : Repérer les valeurs impossibles

**Objectif** : chercher dans `train` et `a_estimer` ce qui **ne peut pas exister**, et le compter. On ne corrige rien ici : on **mesure**.

| Famille de contrôle | Ce qu'on cherche | Attendu |
|---|---|---|
| Valeurs manquantes | cases vides | 0 (l'énoncé l'annonce) |
| Valeurs infinies | `inf` ou `-inf` dans les décimales | 0 |
| Montant | `loss` ≤ 0 | 0 (l'énoncé donne un minimum de 0,67) |
| Identifiants | doublons d'`id`, `id` négatif, `id` présent dans les deux fichiers | 0 |
| Catégorielles | modalité vide ou faite d'espaces, colonne constante | 0 |
| Continues | valeur négative | 0 (l'énoncé annonce un minimum de 0) |

> **Impossible n'est pas rare.** Un très gros sinistre (jusqu'à 121 012) est rare, pas impossible : on ne le supprime pas. C'est la queue de distribution qui décide qui est mal servi, donc ces lignes restent dans les données.

## Bloc 1 : imports et chemins des fichiers

In [9]:
train.head

<bound method NDFrame.head of             id cat1 cat2 cat3 cat4 cat5 cat6 cat7 cat8 cat9  ...     cont6  \
0       585230    B    A    A    B    A    A    A    A    A  ...  0.204548   
1       431675    B    A    A    A    A    B    A    A    A  ...  0.599275   
2       498682    A    B    A    A    A    A    A    A    B  ...  0.760873   
3       456113    B    A    A    A    B    A    A    A    A  ...  0.407919   
4       587001    A    B    A    B    A    A    A    B    B  ...  0.416181   
...        ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...       ...   
150649  380166    A    B    A    B    A    A    A    A    B  ...  0.456654   
150650  494905    A    A    A    A    A    B    A    A    A  ...  0.326207   
150651  148939    A    A    A    A    B    A    A    A    A  ...  0.357412   
150652  222047    A    B    A    A    B    B    A    A    B  ...  0.424491   
150653  408386    B    A    A    A    B    A    A    A    A  ...  0.350956   

           cont7    cont8    cont

In [1]:
# ── Ce que fait ce bloc : importer les outils et trouver où sont les fichiers du sujet 04 ──
from pathlib import Path      # pour manipuler des chemins de fichiers proprement
import numpy as np            # pour détecter les valeurs infinies
import pandas as pd           # pour lire les CSV et manipuler les tableaux (DataFrame)

# Sur Kaggle, les données sont dans /kaggle/input/... ; en local, dans le dossier courant
RACINE_KAGGLE = Path("/kaggle/input")

if RACINE_KAGGLE.exists():
    # On cherche train.csv UNIQUEMENT dans le jeu du sujet 04,
    # pour ne pas lire le train.csv d'une autre compétition
    trouves = [p for p in RACINE_KAGGLE.rglob("train.csv") if "sujet-04" in str(p)]
    if not trouves:
        raise FileNotFoundError("train.csv du sujet 04 introuvable : ajoutez votre jeu (Add Input).")
    DOSSIER = trouves[0].parent     # le dossier qui contient train.csv
else:
    DOSSIER = Path(".")             # en local : le dossier où se trouve le notebook

CHEMIN_TRAIN = DOSSIER / "train.csv"            # sinistres AVEC le montant (loss)
CHEMIN_A_ESTIMER = DOSSIER / "a_estimer.csv"    # sinistres SANS le montant

COLS_CAT = [f"cat{i}" for i in range(1, 117)]    # cat1 ... cat116
COLS_CONT = [f"cont{i}" for i in range(1, 15)]   # cont1 ... cont14

print("Dossier des données  :", DOSSIER)
print("train.csv existe     :", CHEMIN_TRAIN.exists())
print("a_estimer.csv existe :", CHEMIN_A_ESTIMER.exists())

# ── Résultat attendu ──
# Dossier des données  : /kaggle/input/datasets/prcieuxfanou/esgc-vak-sujet-04-fanou   (en local : .)
# train.csv existe     : True
# a_estimer.csv existe : True

Dossier des données  : .
train.csv existe     : True
a_estimer.csv existe : True


## Bloc 2 : lire les deux CSV

Même lecture qu'à l'étape 1.3 : les catégorielles en **texte**, et le texte `NA` conservé comme une vraie modalité. Ici on n'a pas besoin du type `category` : on cherche des anomalies, on ne prépare pas un modèle.

In [10]:
train.head 

<bound method NDFrame.head of             id cat1 cat2 cat3 cat4 cat5 cat6 cat7 cat8 cat9  ...     cont6  \
0       585230    B    A    A    B    A    A    A    A    A  ...  0.204548   
1       431675    B    A    A    A    A    B    A    A    A  ...  0.599275   
2       498682    A    B    A    A    A    A    A    A    B  ...  0.760873   
3       456113    B    A    A    A    B    A    A    A    A  ...  0.407919   
4       587001    A    B    A    B    A    A    A    B    B  ...  0.416181   
...        ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...       ...   
150649  380166    A    B    A    B    A    A    A    A    B  ...  0.456654   
150650  494905    A    A    A    A    A    B    A    A    A  ...  0.326207   
150651  148939    A    A    A    A    B    A    A    A    A  ...  0.357412   
150652  222047    A    B    A    A    B    B    A    A    B  ...  0.424491   
150653  408386    B    A    A    A    B    A    A    A    A  ...  0.350956   

           cont7    cont8    cont

In [2]:
# ── Ce que fait ce bloc : lire train et a_estimer, avec les types de l'étape 1.3 ──
types_lecture = {c: str for c in COLS_CAT}   # les 116 cat lues comme du texte

# keep_default_na=False : le texte "NA" reste du texte ; seule une case vraiment vide devient manquante
train = pd.read_csv(CHEMIN_TRAIN, dtype=types_lecture, keep_default_na=False, na_values=[""])
a_estimer = pd.read_csv(CHEMIN_A_ESTIMER, dtype=types_lecture, keep_default_na=False, na_values=[""])

# Les colonnes numériques en décimal, id en entier
for df in (train, a_estimer):
    df["id"] = df["id"].astype("int64")
    for c in COLS_CONT:
        df[c] = df[c].astype("float64")
train["loss"] = train["loss"].astype("float64")

print("Forme de train     :", train.shape)
print("Forme de a_estimer :", a_estimer.shape)

# ── Résultat attendu ──
# Forme de train     : (150654, 132)
# Forme de a_estimer : (37664, 131)

Forme de train     : (150654, 132)
Forme de a_estimer : (37664, 131)


## Bloc 3 : valeurs manquantes et modalités vides

In [3]:
# ── Ce que fait ce bloc : compter les cases vides et les modalités faites d'espaces ──
# Valeurs manquantes : case vide dans le fichier
manquantes_train = int(train.isna().sum().sum())
manquantes_a_estimer = int(a_estimer.isna().sum().sum())

# Modalités "invisibles" : un texte fait uniquement d'espaces (il ressemble à du vide mais n'est pas NaN)
def compter_vides(df):
    total = 0
    for c in COLS_CAT:
        total += int(df[c].str.strip().eq("").sum())   # strip() enlève les espaces ; eq("") teste le vide
    return total

vides_train = compter_vides(train)
vides_a_estimer = compter_vides(a_estimer)

print("Valeurs manquantes dans train     :", manquantes_train)
print("Valeurs manquantes dans a_estimer :", manquantes_a_estimer)
print("Modalités vides ou d'espaces dans train     :", vides_train)
print("Modalités vides ou d'espaces dans a_estimer :", vides_a_estimer)

# ── Résultat attendu ──
# 0 partout (l'énoncé annonce « aucune valeur manquante »). Une valeur non nulle serait à noter dans le journal.

Valeurs manquantes dans train     : 0
Valeurs manquantes dans a_estimer : 0
Modalités vides ou d'espaces dans train     : 0
Modalités vides ou d'espaces dans a_estimer : 0


## Bloc 4 : valeurs infinies et plages des continues

In [4]:
# ── Ce que fait ce bloc : chercher les infinis et lire les minimum / maximum de chaque continue ──
# Infinis : np.isinf détecte inf et -inf
infinies_train = int(np.isinf(train[COLS_CONT + ["loss"]]).sum().sum())
infinies_a_estimer = int(np.isinf(a_estimer[COLS_CONT]).sum().sum())

print("Valeurs infinies dans train (cont + loss) :", infinies_train)
print("Valeurs infinies dans a_estimer (cont)    :", infinies_a_estimer)

# Plage de chaque continue, dans les deux fichiers : on LIT, on ne suppose pas
plages = pd.DataFrame({
    "min_train": train[COLS_CONT].min(),
    "max_train": train[COLS_CONT].max(),
    "min_a_estimer": a_estimer[COLS_CONT].min(),
    "max_a_estimer": a_estimer[COLS_CONT].max(),
})
display(plages)

print("Plus petite valeur des continues (train)    :", float(plages["min_train"].min()))
print("Plus grande valeur des continues (train)    :", float(plages["max_train"].max()))

# ── Résultat attendu ──
# 0 infini. L'énoncé annonce des continues « de 0 à 587 602 » : comparez avec le tableau,
# sans le supposer. Un écart entre l'énoncé et la mesure est une information à noter dans le journal.

Valeurs infinies dans train (cont + loss) : 0
Valeurs infinies dans a_estimer (cont)    : 0


,min_train,max_train,min_a_estimer,max_a_estimer
cont1,0.000016,0.984975,0.000016,0.984452
cont2,0.001149,0.862654,0.001149,0.862654
cont3,0.002634,0.944251,0.002634,0.944251
cont4,0.176921,0.954297,0.176921,0.952482
cont5,0.281143,0.982520,0.281143,0.983674
cont6,0.012683,0.997162,0.012683,0.997162
cont7,0.069503,1.000000,0.069503,1.000000
cont8,0.236880,0.980200,0.236880,0.980200
cont9,0.000080,0.995400,0.000080,0.993790
cont10,0.000000,0.994980,0.000000,0.994630


Plus petite valeur des continues (train)    : 0.0
Plus grande valeur des continues (train)    : 1.0


## Bloc 5 : le montant `loss`

Un montant de sinistre doit être **strictement positif**. On vérifie aussi que le minimum et le maximum correspondent à ceux de l'énoncé (0,67 et 121 012).

In [5]:
# ── Ce que fait ce bloc : contrôler que loss est un montant possible ──
loss = train["loss"]

nb_loss_negatif_ou_nul = int((loss <= 0).sum())    # un montant <= 0 est impossible ici

print("loss <= 0           :", nb_loss_negatif_ou_nul)
print("loss minimum        :", round(float(loss.min()), 2))
print("loss maximum        :", round(float(loss.max()), 2))
print("Nombre de lignes    :", len(loss))

# Quelques repères sur la distribution (pour LIRE la queue, pas pour supprimer quoi que ce soit)
display(loss.describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_frame("loss"))

# ── Résultat attendu ──
# loss <= 0 : 0 ; minimum : 0.67 ; maximum : environ 121012.
# La médiane est bien inférieure à la moyenne : c'est la queue de distribution (étape 1.10).

loss <= 0           : 0
loss minimum        : 0.67
loss maximum        : 121012.25
Nombre de lignes    : 150654


,loss
count,150654.00
mean,3038.92
std,2915.01
min,0.67
50%,2115.57
90%,6401.73
99%,14025.82
max,121012.25


## Bloc 6 : identifiants

In [6]:
# ── Ce que fait ce bloc : vérifier que chaque sinistre a un id propre et unique ──
doublons_id_train = int(train["id"].duplicated().sum())          # même id sur 2 lignes de train
doublons_id_a_estimer = int(a_estimer["id"].duplicated().sum())  # même id sur 2 lignes de a_estimer
id_negatifs = int((train["id"] < 0).sum() + (a_estimer["id"] < 0).sum())

# Un id présent dans les DEUX fichiers voudrait dire que le même sinistre est à la fois
# connu (train) et à estimer (a_estimer) : mélange des fichiers
id_communs = len(set(train["id"]) & set(a_estimer["id"]))

print("id en double dans train     :", doublons_id_train)
print("id en double dans a_estimer :", doublons_id_a_estimer)
print("id négatifs (2 fichiers)    :", id_negatifs)
print("id présents dans les 2 fichiers :", id_communs)

# ── Résultat attendu ──
# 0 partout.

id en double dans train     : 0
id en double dans a_estimer : 0
id négatifs (2 fichiers)    : 0
id présents dans les 2 fichiers : 0


## Bloc 7 : lignes dupliquées, colonnes constantes, continues hors plage

Ici, ce sont des **signaux à lire**, pas forcément des erreurs : deux sinistres peuvent avoir le même dossier sans que ce soit une faute. On compte pour documenter, et la décision viendra plus tard.

In [7]:
# ── Ce que fait ce bloc : repérer trois signaux faibles ──
# 1) Colonnes cat constantes : une seule modalité (l'énoncé annonce au moins 2 par colonne)
nb_cat_constantes = int(sum(train[c].nunique() < 2 for c in COLS_CAT))

# 2) Lignes dont TOUTES les variables explicatives sont identiques (on ignore id et loss)
variables = COLS_CAT + COLS_CONT
lignes_dupliquees_train = int(train[variables].duplicated().sum())
lignes_dupliquees_a_estimer = int(a_estimer[variables].duplicated().sum())

# 3) Continues négatives, et valeurs de a_estimer en dehors de la plage vue dans train
cont_negatives = int((train[COLS_CONT] < 0).sum().sum() + (a_estimer[COLS_CONT] < 0).sum().sum())

hors_plage = 0
for c in COLS_CONT:
    bas, haut = train[c].min(), train[c].max()
    hors_plage += int(((a_estimer[c] < bas) | (a_estimer[c] > haut)).sum())

print("Colonnes cat constantes (train)                  :", nb_cat_constantes)
print("Lignes à variables identiques (train)            :", lignes_dupliquees_train)
print("Lignes à variables identiques (a_estimer)        :", lignes_dupliquees_a_estimer)
print("Valeurs cont négatives (2 fichiers)              :", cont_negatives)
print("Valeurs de a_estimer hors de la plage de train   :", hors_plage)

# ── Résultat attendu ──
# Colonnes constantes : 0 ; continues négatives : 0.
# Les lignes dupliquées et les valeurs hors plage sont des MESURES : notez-les, ne les supposez pas.

Colonnes cat constantes (train)                  : 0
Lignes à variables identiques (train)            : 1
Lignes à variables identiques (a_estimer)        : 0
Valeurs cont négatives (2 fichiers)              : 0
Valeurs de a_estimer hors de la plage de train   : 3


## Bloc 8 : tableau récapitulatif de l'étape 1.5

In [8]:
# ── Ce que fait ce bloc : regrouper tous les contrôles dans un seul tableau ──
controles = [
    ("Valeurs manquantes (train)",                    0, manquantes_train),
    ("Valeurs manquantes (a_estimer)",                0, manquantes_a_estimer),
    ("Modalités vides ou d'espaces (train)",          0, vides_train),
    ("Modalités vides ou d'espaces (a_estimer)",      0, vides_a_estimer),
    ("Valeurs infinies (train)",                      0, infinies_train),
    ("Valeurs infinies (a_estimer)",                  0, infinies_a_estimer),
    ("loss <= 0",                                     0, nb_loss_negatif_ou_nul),
    ("id en double (train)",                          0, doublons_id_train),
    ("id en double (a_estimer)",                      0, doublons_id_a_estimer),
    ("id négatifs",                                   0, id_negatifs),
    ("id présents dans les deux fichiers",            0, id_communs),
    ("Colonnes cat constantes",                       0, nb_cat_constantes),
    ("Valeurs cont négatives",                        0, cont_negatives),
    ("loss minimum (énoncé : 0,67)",                  0.67,   round(float(train["loss"].min()), 2)),
    ("loss maximum, partie entière (énoncé : 121 012)", 121012, int(train["loss"].max())),
]

recap = pd.DataFrame(controles, columns=["Contrôle", "Attendu", "Obtenu"])
recap["Verdict"] = recap.apply(
    lambda ligne: "OK" if ligne["Obtenu"] == ligne["Attendu"] else "ATTENTION", axis=1
)
display(recap)

# Signaux à documenter (ce ne sont pas des contrôles)
print("À signaler : lignes à variables identiques :", lignes_dupliquees_train, "(train),",
      lignes_dupliquees_a_estimer, "(a_estimer)")
print("À signaler : valeurs de a_estimer hors de la plage de train :", hors_plage)

if (recap["Verdict"] == "OK").all():
    print("Étape 1.5 validée : aucune valeur impossible détectée.")
else:
    print("ATTENTION : au moins un contrôle ne passe pas. Regarder la colonne Verdict.")

# ── Résultat attendu ──
# Un tableau de 15 lignes, toutes en OK, puis :
# Étape 1.5 validée : aucune valeur impossible détectée.
# (Un écart sur le minimum ou le maximum de loss est une information à noter dans le journal.)

,Contrôle,Attendu,Obtenu,Verdict
0,Valeurs manquantes (train),0.00,0.00,OK
1,Valeurs manquantes (a_estimer),0.00,0.00,OK
2,Modalités vides ou d'espaces (train),0.00,0.00,OK
3,Modalités vides ou d'espaces (a_estimer),0.00,0.00,OK
4,Valeurs infinies (train),0.00,0.00,OK
5,Valeurs infinies (a_estimer),0.00,0.00,OK
6,loss <= 0,0.00,0.00,OK
7,id en double (train),0.00,0.00,OK
8,id en double (a_estimer),0.00,0.00,OK
9,id négatifs,0.00,0.00,OK


À signaler : lignes à variables identiques : 1 (train), 0 (a_estimer)
À signaler : valeurs de a_estimer hors de la plage de train : 3
Étape 1.5 validée : aucune valeur impossible détectée.
